# Gen Set Enrichment Analysis with GSEApy for TCGA LUAD LOY vs ROY

### Import

In [ ]:
import os
import pickle

import pandas as pd
import seaborn as sns
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import plotly.express as px
import scipy.cluster.hierarchy as sch

%matplotlib inline 

from gseapy import barplot, dotplot
from gseapy import gseaplot


from gseapy import Biomart
bm = Biomart()

In [41]:
# Set directories
data_dir = '/data'

In [ ]:
# Load data after DESeq2 analysis
DE_LOY_ROY = pd.read_csv(os.path.join(data_dir,'DE_TCGA_LUAD_LOY_vs_ROY.csv'), sep=',', header=0, index_col=0)
DE_LOY_ROY = DE_LOY_ROY.reset_index().rename(columns={"index": "ensembl_gene_id"})

In [ ]:
# gene_map id from biomart do not include version suffiix, there for we need to strip it before merging
DE_LOY_ROY['ensembl_gene_id'] = DE_LOY_ROY['ensembl_gene_id'].str.split('.').str[0]

In [ ]:
# Query biomart for mapping Ensembl ID to gene name (HGNC symbol)
gene_map = bm.query(
    dataset='hsapiens_gene_ensembl',  # human
    attributes=['ensembl_gene_id', 'hgnc_symbol']
)
gene_map

In [ ]:
# Merge expression data with gene names
DE_LOY_ROY = pd.merge(DE_LOY_ROY, gene_map, on="ensembl_gene_id", how="left")
DE_LOY_ROY

In [ ]:
# Load gene sets
MSigDB_Hallmark_2020 = pd.read_csv(os.path.join(data_dir,'MSigDB_Hallmark_2020.csv'), delimiter=';', header=None) # can be downloaded from: https://www.gsea-msigdb.org/gsea/msigdb/human/collections.jsp#H

In [45]:
# convert to dictionary format
hall_dict = {}
for _, row in MSigDB_Hallmark_2020.iterrows():
    # Skip rows where the first column is NaN or empty
    if pd.notna(row[0]) and row[0].strip():
        hall_dict[row[0]] = list(row[2:].dropna().astype(str))

### Prerank API

In [ ]:
# prepare ranked gene list
# sort by absolute log2FoldChange
sorted_gene_list = DE_LOY_ROY.sort_values(by='log2FoldChange', ascending=False)

# Subset the DataFrame to include only the gene name column
glist_rnk = sorted_gene_list[['hgnc_symbol', 'log2FoldChange']]

# Set 'gene_name' as the index
glist_rnk.set_index('hgnc_symbol', inplace=True)

In [ ]:
pre_res_hallmarks = gp.prerank(rnk=glist_rnk, 
                     gene_sets=hall_dict, 
                     threads=4,
                     min_size=15, 
                     max_size=500, 
                     permutation_num=1000, 
                     outdir=output_dir,
                     seed=6,
                     verbose=True, 
                    )

### Dotplot - Fig 2I

In [ ]:
# dotplot (Fig 2I)
plt.style.use("default")

ax = dotplot(pre_res_hallmarks.res2d,
             column="FDR q-val",
             cmap=plt.cm.viridis,
             size=6, # adjust dot size
             top_term = 10,  
             figsize=(3,7), show_ring=True, cutoff=0.1)

ax.set_title('TCGA LUAD LOY vs ROY', fontsize=12)
ax.set_xlabel('NES', fontsize=12)
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)

fig = ax.figure
fig.savefig(
    "TCGA_LOY_vs_ROY_dotplot.pdf",
    format="pdf",
    bbox_inches="tight"
)

fig.savefig(
    "TCGA_LOY_vs_ROY_dotplot.tiff",
    format="tiff",
    dpi=300,
    bbox_inches="tight"
)